# Interactive Visualizations

This notebook creates interactive choropleth maps to visualize refugee and asylum seeker flows involving Turkey. Two maps are generated:
1. **Applications TO Turkey** - Shows origin countries of refugees seeking asylum in Turkey
2. **Applications FROM Turkey** - Shows destination countries where Turkish citizens seek asylum

Both maps feature year-by-year animation sliders and include recognition rate data in hover tooltips.

**Note:** For detailed analysis, data limitations, and key observations about Turkey's dual role as both refugee host and origin country, see Notebook 02 (Analysis). This notebook focuses on creating interactive visualizations of the findings.

In [26]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [27]:
clean_data_dir = Path('../data/clean')

apps_to_turkey = pd.read_csv(clean_data_dir / 'applications_to_turkey_clean.csv')
apps_from_turkey = pd.read_csv(clean_data_dir / 'applications_from_turkey_clean.csv')
decisions_turkey = pd.read_csv(clean_data_dir / 'decisions_from_turkey_clean.csv')
decisions_from_others = pd.read_csv(clean_data_dir / 'decisions_from_other_countries_clean.csv')

print('Data loaded successfully!')

visualizations_dir = Path('../visualizations')
visualizations_dir.mkdir(exist_ok=True)
print(f"Visualizations will be saved to: {visualizations_dir}")

Data loaded successfully!
Visualizations will be saved to: ..\visualizations


## Map 1: Asylum Applications TO Turkey

### Data Preparation

Combining application data with recognition rates by country and year. Data is filtered to 2000-2018 due to Turkey's shift to "Temporary Protection" status after 2018, which created incomplete formal asylum application records.

In [28]:
apps_to_turkey_by_country = apps_to_turkey.groupby(['Country of Origin Name', 'Country of Origin Code', 'Year'])['Number of Applications'].sum().reset_index()

recognition_data_turkey = decisions_turkey.groupby(['Country of Origin Name', 'Year']).agg({
    'Recognized': 'sum',
    'Total Decided': 'sum'
}).reset_index()

recognition_data_turkey['Recognition Rate'] = (
    recognition_data_turkey['Recognized'] / recognition_data_turkey['Total Decided'] * 100
).round(1)


apps_to_turkey_by_country = apps_to_turkey_by_country.merge(
    recognition_data_turkey[['Country of Origin Name', 'Year', 'Recognition Rate']], 
    on=['Country of Origin Name', 'Year'], 
    how='left'  # Left join so we keep all applications even if no decision data
)

apps_to_turkey_by_country['Recognition Rate'] = apps_to_turkey_by_country['Recognition Rate'].apply(
    lambda x: f"{x:.1f}%" if pd.notna(x) else 'No data'
)

apps_to_turkey_by_country.columns = ['Country', 'ISO_Code', 'Year', 'Applications', 'Recognition Rate']
apps_to_turkey_by_country = apps_to_turkey_by_country.sort_values(['Year', 'Country']).reset_index(drop=True)

# Cut off at 2018 because of the Temporary Protection status of migrants after 2018 which caused data incompleteness.
apps_to_turkey_by_country_filtered = apps_to_turkey_by_country[apps_to_turkey_by_country['Year'] <= 2018]

print(apps_to_turkey_by_country.sample(5))

                Country ISO_Code  Year  Applications Recognition Rate
291          Bangladesh      BGD  2011            11             0.0%
120  State of Palestine      PSE  2005            29             0.0%
0           Afghanistan      AFG  2000            81            28.4%
79              Algeria      DZA  2004             7             0.0%
520        Turkmenistan      TKM  2015           152             0.0%


### Interactive Choropleth Map - Applications TO Turkey

This map shows countries colored by the number of asylum applications sent to Turkey. Use the slider to see how refugee flows changed over time, particularly during the Syrian crisis (2011-2015). Turkey is highlighted in red as the destination country.

In [29]:
fig = px.choropleth(
    apps_to_turkey_by_country_filtered,
    locations='ISO_Code',                 
    color='Applications',              
    hover_name='Country',
    animation_frame='Year',
    hover_data={'ISO_Code': False,
                'Year': False,
                'Applications': ':,',
                'Recognition Rate': True
                },
    color_continuous_scale='Viridis',       
    labels={'Applications': 'Applications',
            'Recognition Rate': 'Recognition Rate'},
    title='Asylum Applications to Turkey by Origin Country (2000-2018)'
)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    showcountries=True
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    height=600
)

fig.update_coloraxes(
    colorbar=dict(
        title="Number of<br>Applications",
        thickness=15,
        len=0.7
    )
)

fig.add_trace(
    go.Choropleth(
        locations=['TUR'],
        z=[0],
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],  # Transparent fill
        showscale=False,
        marker_line_color='red',
        marker_line_width=2,
        text=['Turkey (Destination)'],
        hoverinfo='text',
    )
)

fig.add_annotation(
    text="Source: UNHCR Population Statistics Database",
    xref="paper", yref="paper",
    x=0, y=-0.1,
    showarrow=False,
    font=dict(size=10, color="gray")
)

fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 1000
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 500

fig.show()

fig.write_html(visualizations_dir / "turkey_asylum_applications_map.html")
print("Map saved successfully!")

Map saved successfully!


## Map 2: Asylum Applications FROM Turkey

### Data Preparation

Preparing data for Turkish asylum seekers by destination country. Unlike the previous map, this includes all years (2000-2025) as the data is more complete for outbound applications.

In [30]:
apps_from_turkey_by_country = apps_from_turkey.groupby(['Country of Asylum Name', 'Country of Asylum Code', 'Year'])['Number of Applications'].sum().reset_index()

recognition_data_others = decisions_from_others.groupby(['Country of Asylum Name', 'Year']).agg({
    'Recognized': 'sum',
    'Total Decided': 'sum'
}).reset_index()

recognition_data_others['Recognition Rate'] = (
    recognition_data_others['Recognized'] / recognition_data_others['Total Decided'] * 100
).round(1)

apps_from_turkey_by_country = apps_from_turkey_by_country.merge(
    recognition_data_others[['Country of Asylum Name', 'Year', 'Recognition Rate']],
    on=['Country of Asylum Name', 'Year'],
    how='left'
)

apps_from_turkey_by_country['Recognition Rate'] = apps_from_turkey_by_country['Recognition Rate'].apply(
    lambda x: f"{x:.1f}%" if pd.notna(x) else 'No Data'
)

apps_from_turkey_by_country.columns = ['Country', 'ISO_Code', 'Year', 'Applications', 'Recognition Rate']
apps_from_turkey_by_country = apps_from_turkey_by_country.sort_values(['Year', 'Country']).reset_index(drop=True) # Sort by year first so the map slider is accurate

print(apps_from_turkey_by_country.sample(5))

       Country ISO_Code  Year  Applications Recognition Rate
297   Bulgaria      BGR  2008             7             0.0%
73     Austria      AUT  2002          3561             2.9%
1037     Libya      LBY  2022             5           100.0%
402    Armenia      ARM  2011             5             0.0%
192    Czechia      CZE  2005            33            19.2%


### Interactive Choropleth Map - Applications FROM Turkey

This map shows where Turkish citizens sought asylum, with a notable surge after 2016 (failed coup attempt) and continuing through 2023. Turkey is highlighted in orange as the origin country.

In [32]:
fig2 = px.choropleth(
    apps_from_turkey_by_country,
    locations='ISO_Code',
    color='Applications',
    hover_name='Country',
    animation_frame='Year',
    hover_data={'ISO_Code': False,
                'Year': False,
                'Applications': ':,',
                'Recognition Rate': True
                },
    color_continuous_scale='Viridis',
    labels={'Applications': 'Applications',
            'Recognition Rate': 'Recognition Rate'},
    title='Asylum Applications from Turkey by Destination Country (2000-2025)'
)

fig2.update_geos(
    visible=False,
    showcountries=True
)

fig2.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    height=600
)

fig2.update_coloraxes(
    colorbar=dict(
        title="Number of<br>Applications",
        thickness=15,
        len=0.7
    )
)

fig2.add_trace(
    go.Choropleth(
        locations=['TUR'],
        z=[0],
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],  # Transparent fill
        showscale=False,
        marker_line_color='orange',
        marker_line_width=2,
        text=['Turkey (Origin)'],
        hoverinfo='text',
    )
)

fig2.add_annotation(
    text="Source: UNHCR Population Statistics Database",
    xref="paper", yref="paper",
    x=0, y=-0.1,
    showarrow=False,
    font=dict(size=10, color="gray")
)

fig2.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 1000
fig2.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 500

fig2.show()

fig2.write_html(visualizations_dir / "others_asylum_applications_map.html")
print("Map saved successfully!")

Map saved successfully!


## Summary

Two interactive maps have been created and saved to the `/visualizations` directory:
- `turkey_asylum_applications_map.html` - Refugees coming TO Turkey (2000-2018)
- `others_asylum_applications_map.html` - Turkish asylum seekers going abroad (2000-2025)

Both maps are fully interactive with year sliders and can be opened in any web browser.